# NYC Subway Fleet Reliability & MDBF Analysis

This notebook analyzes fleet reliability through Mean Distance Between Failures (MDBF), terminal
on-time performance (OTP), major incident patterns, and service delivery metrics. We build a
predictive model for monthly MDBF using operational features.

**Data Sources:**
- MDBF by car class (monthly, from MTA performance dashboards)
- Terminal on-time performance by division (monthly)
- Major incident logs (categorized by type)
- Service delivered as % of scheduled (by line, monthly)

In [ ]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 120

%matplotlib inline

In [ ]:
# ---- Configuration ----
# Replace YOUR_ACCOUNT_ID with your actual AWS account ID
ANALYTICS_BUCKET = "railtime-analytics-YOUR_ACCOUNT_ID"

MDBF_PATH = f"s3://{ANALYTICS_BUCKET}/historical/mdbf/"
OTP_PATH = f"s3://{ANALYTICS_BUCKET}/historical/terminal_otp/"
INCIDENTS_PATH = f"s3://{ANALYTICS_BUCKET}/historical/major_incidents/"
SERVICE_DELIVERED_PATH = f"s3://{ANALYTICS_BUCKET}/historical/service_delivered/"

# Division mappings
DIVISION_A_LINES = ["1", "2", "3", "4", "5", "6", "7", "S"]
DIVISION_B_LINES = ["A", "C", "E", "B", "D", "F", "M", "G", "J", "Z", "L", "N", "Q", "R", "W"]

# Car class to division mapping
CAR_CLASS_DIVISION = {
    "R142": "A", "R142A": "A", "R188": "A",
    "R143": "B", "R160": "B", "R179": "B", "R211": "B",
    "R62": "A", "R62A": "A", "R68": "B", "R68A": "B",
}

In [ ]:
# ---- Load Data ----
mdbf_df = wr.s3.read_parquet(MDBF_PATH)
otp_df = wr.s3.read_parquet(OTP_PATH)
incidents_df = wr.s3.read_parquet(INCIDENTS_PATH)
service_df = wr.s3.read_parquet(SERVICE_DELIVERED_PATH)

datasets = {
    "MDBF": mdbf_df,
    "Terminal OTP": otp_df,
    "Major Incidents": incidents_df,
    "Service Delivered": service_df,
}

for name, d in datasets.items():
    print(f"\n{'=' * 40}")
    print(f"{name}: shape={d.shape}")
    print(f"Columns: {list(d.columns)}")
    print(f"Date range: {d.iloc[:, 0].min()} to {d.iloc[:, 0].max()}" if len(d) > 0 else "Empty")
    print(d.head(3))

## Fleet Reliability: MDBF Trends

Mean Distance Between Failures (MDBF) measures fleet reliability in miles traveled between
mechanical breakdowns. Higher MDBF = more reliable cars.

In [ ]:
# ---- MDBF Trends by Car Class ----
mdbf = mdbf_df.copy()
mdbf["date"] = pd.to_datetime(mdbf["date"])

# Identify car class column
car_class_col = [c for c in mdbf.columns if "car" in c.lower() or "class" in c.lower() or "fleet" in c.lower()]
car_class_col = car_class_col[0] if car_class_col else "car_class"

mdbf_col = [c for c in mdbf.columns if "mdbf" in c.lower() or "miles" in c.lower()]
mdbf_col = mdbf_col[0] if mdbf_col else "mdbf"

car_classes = mdbf[car_class_col].unique()
n_classes = len(car_classes)

# Color palette
colors = sns.color_palette("husl", n_classes)

fig, ax = plt.subplots(figsize=(16, 8))

for i, car_class in enumerate(sorted(car_classes)):
    subset = mdbf[mdbf[car_class_col] == car_class].sort_values("date")
    # 3-month rolling average for smoother trend
    subset["mdbf_smooth"] = subset[mdbf_col].rolling(3, min_periods=1).mean()
    ax.plot(
        subset["date"], subset["mdbf_smooth"],
        linewidth=2, color=colors[i], label=car_class, marker="o", markersize=2
    )

ax.set_title("MDBF Trends by Car Class (3-Month Rolling Average)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Mean Distance Between Failures (miles)")
ax.legend(title="Car Class", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

# Summary statistics
summary = mdbf.groupby(car_class_col)[mdbf_col].agg(["mean", "median", "std", "min", "max"]).round(0)
summary = summary.sort_values("mean", ascending=False)
print("\nMDBF Summary by Car Class:")
print(summary)

In [ ]:
# ---- Division A vs Division B: MDBF Comparison ----
mdbf["division"] = mdbf[car_class_col].map(CAR_CLASS_DIVISION)

# Drop any car classes not in our mapping
mdbf_div = mdbf.dropna(subset=["division"])

div_a = mdbf_div.loc[mdbf_div["division"] == "A", mdbf_col]
div_b = mdbf_div.loc[mdbf_div["division"] == "B", mdbf_col]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot
sns.boxplot(
    data=mdbf_div, x="division", y=mdbf_col,
    palette={"A": "#1f77b4", "B": "#ff7f0e"},
    ax=axes[0], fliersize=2
)
axes[0].set_title("MDBF: Division A vs Division B", fontweight="bold")
axes[0].set_xlabel("Division")
axes[0].set_ylabel("MDBF (miles)")

# Violin plot with individual points
sns.violinplot(
    data=mdbf_div, x="division", y=mdbf_col,
    palette={"A": "#1f77b4", "B": "#ff7f0e"},
    ax=axes[1], inner="quartile", alpha=0.7
)
axes[1].set_title("MDBF Distribution (Violin)", fontweight="bold")
axes[1].set_xlabel("Division")
axes[1].set_ylabel("MDBF (miles)")

plt.tight_layout()
plt.show()

# Welch's t-test
t_stat, p_value = stats.ttest_ind(div_a, div_b, equal_var=False)
print(f"\nWelch's t-test: Division A vs Division B")
print(f"  Division A mean MDBF: {div_a.mean():,.0f} miles (n={len(div_a)})")
print(f"  Division B mean MDBF: {div_b.mean():,.0f} miles (n={len(div_b)})")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Significant at alpha=0.05: {'Yes' if p_value < 0.05 else 'No'}")

## Incident Analysis

Major incidents are categorized by type (signal failures, car equipment failures, track defects,
power issues, etc.) and analyzed for temporal and spatial patterns.

In [ ]:
# ---- Incident Categories Over Time ----
incidents = incidents_df.copy()
incidents["date"] = pd.to_datetime(incidents["date"])
incidents["year_month"] = incidents["date"].dt.to_period("M")

# Identify category column
cat_col = [c for c in incidents.columns if "categ" in c.lower() or "type" in c.lower()]
cat_col = cat_col[0] if cat_col else "category"

# Count incidents by category and month
monthly_incidents = incidents.groupby(["year_month", cat_col]).size().reset_index(name="count")
pivot = monthly_incidents.pivot(index="year_month", columns=cat_col, values="count").fillna(0)

fig, ax = plt.subplots(figsize=(16, 8))

pivot.plot(
    kind="bar", stacked=True, ax=ax,
    colormap="tab10", edgecolor="none", width=0.8
)

ax.set_title("Major Incidents by Category (Monthly)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Incidents")
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)

# Show every 6th x-tick label to reduce crowding
tick_labels = [str(label) if i % 6 == 0 else "" for i, label in enumerate(pivot.index)]
ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)

ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# Total by category
print("\nTotal incidents by category:")
category_totals = incidents[cat_col].value_counts()
for cat, count in category_totals.items():
    print(f"  {cat}: {count}")

In [ ]:
# ---- Incident Frequency Heatmap by Line ----
line_col = [c for c in incidents.columns if "line" in c.lower() or "route" in c.lower()]
line_col = line_col[0] if line_col else "line"

# Count incidents per (line, category)
line_cat = incidents.groupby([line_col, cat_col]).size().reset_index(name="count")
heatmap_data = line_cat.pivot(index=line_col, columns=cat_col, values="count").fillna(0)

# Sort lines by total incidents
heatmap_data["total"] = heatmap_data.sum(axis=1)
heatmap_data = heatmap_data.sort_values("total", ascending=False).drop(columns="total")

fig, ax = plt.subplots(figsize=(14, 10))

sns.heatmap(
    heatmap_data, annot=True, fmt=".0f", cmap="YlOrRd",
    linewidths=0.5, linecolor="white", ax=ax,
    cbar_kws={"label": "Incident Count"}
)

ax.set_title("Incident Frequency by Line and Category", fontsize=14, fontweight="bold")
ax.set_xlabel("Incident Category")
ax.set_ylabel("Line")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Service Delivery

Percentage of scheduled service actually delivered, broken out by line and weekday vs weekend.

In [ ]:
# ---- Service Delivered by Line ----
service = service_df.copy()
service["date"] = pd.to_datetime(service["date"])

svc_line_col = [c for c in service.columns if "line" in c.lower() or "route" in c.lower()]
svc_line_col = svc_line_col[0] if svc_line_col else "line"

svc_pct_col = [c for c in service.columns if "pct" in c.lower() or "percent" in c.lower() or "delivered" in c.lower()]
svc_pct_col = svc_pct_col[0] if svc_pct_col else "service_pct"

day_type_col = [c for c in service.columns if "day" in c.lower() and "type" in c.lower()]
if day_type_col:
    day_type_col = day_type_col[0]
else:
    # Derive from date
    day_type_col = "day_type"
    service[day_type_col] = np.where(
        service["date"].dt.dayofweek >= 5, "Weekend", "Weekday"
    )

# Average service % by line
line_avg = service.groupby(svc_line_col)[svc_pct_col].mean().sort_values(ascending=False)

# Weekday vs Weekend by line
line_daytype = service.groupby([svc_line_col, day_type_col])[svc_pct_col].mean().reset_index()
line_daytype_pivot = line_daytype.pivot(index=svc_line_col, columns=day_type_col, values=svc_pct_col)
line_daytype_pivot = line_daytype_pivot.loc[line_avg.index]  # Same sort order

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Overall bar chart
colors_bar = ["#2ecc71" if v >= 95 else "#f39c12" if v >= 90 else "#e74c3c" for v in line_avg.values]
line_avg.plot(kind="bar", ax=axes[0], color=colors_bar, edgecolor="none")
axes[0].axhline(y=95, color="green", linestyle="--", alpha=0.5, label="95% target")
axes[0].axhline(y=90, color="orange", linestyle="--", alpha=0.5, label="90% threshold")
axes[0].set_title("Average % Service Delivered by Line", fontweight="bold")
axes[0].set_ylabel("% Service Delivered")
axes[0].set_xlabel("Line")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(fontsize=9)
axes[0].set_ylim(bottom=min(line_avg.min() - 5, 80))

# Weekday vs Weekend grouped bar
line_daytype_pivot.plot(kind="bar", ax=axes[1], color=["steelblue", "coral"], edgecolor="none")
axes[1].set_title("Service Delivered: Weekday vs Weekend", fontweight="bold")
axes[1].set_ylabel("% Service Delivered")
axes[1].set_xlabel("Line")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Day Type")
axes[1].set_ylim(bottom=min(line_daytype_pivot.min().min() - 5, 80))

plt.tight_layout()
plt.show()

## On-Time Performance

Terminal on-time performance (OTP) measures the percentage of trains arriving at their terminal
station within a defined threshold of the scheduled time.

In [ ]:
# ---- OTP Trends by Division ----
otp = otp_df.copy()
otp["date"] = pd.to_datetime(otp["date"])

otp_div_col = [c for c in otp.columns if "div" in c.lower()]
otp_div_col = otp_div_col[0] if otp_div_col else "division"

otp_val_col = [c for c in otp.columns if "otp" in c.lower() or "on_time" in c.lower() or "pct" in c.lower()]
otp_val_col = otp_val_col[0] if otp_val_col else "otp_pct"

fig, ax = plt.subplots(figsize=(16, 7))

division_colors = {"A": "#1f77b4", "B": "#ff7f0e", "Division A": "#1f77b4", "Division B": "#ff7f0e"}

for div_name in sorted(otp[otp_div_col].unique()):
    subset = otp[otp[otp_div_col] == div_name].sort_values("date")
    # 3-month rolling average
    subset["otp_smooth"] = subset[otp_val_col].rolling(3, min_periods=1).mean()
    color = division_colors.get(div_name, None)
    ax.plot(
        subset["date"], subset[otp_val_col],
        alpha=0.25, linewidth=0.5, color=color
    )
    ax.plot(
        subset["date"], subset["otp_smooth"],
        linewidth=2, color=color, label=f"{div_name} (3-mo avg)"
    )

ax.axhline(y=80, color="red", linestyle="--", alpha=0.5, label="80% target")
ax.set_title("Terminal On-Time Performance by Division", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("OTP (%)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

# Summary
otp_summary = otp.groupby(otp_div_col)[otp_val_col].describe().round(1)
print("\nOTP Summary by Division:")
print(otp_summary)

## Predictive Model: Monthly MDBF

We join MDBF with OTP, incident counts, and service delivery to build a Random Forest model
that predicts monthly MDBF from operational features.

In [ ]:
# ---- Prepare Joint Dataset ----

# Aggregate MDBF to monthly by division
mdbf_monthly = mdbf.copy()
mdbf_monthly["year_month"] = mdbf_monthly["date"].dt.to_period("M")
mdbf_monthly = mdbf_monthly.groupby(["year_month", "division"]).agg(
    mdbf_mean=(mdbf_col, "mean"),
    mdbf_min=(mdbf_col, "min"),
    mdbf_max=(mdbf_col, "max"),
    n_car_classes=(car_class_col, "nunique"),
).reset_index()

# Aggregate OTP to monthly by division
otp_monthly = otp.copy()
otp_monthly["year_month"] = otp_monthly["date"].dt.to_period("M")
# Map division column to A/B if needed
otp_div_mapping = {v: v for v in otp_monthly[otp_div_col].unique()}
if "Division A" in otp_div_mapping:
    otp_div_mapping = {"Division A": "A", "Division B": "B"}
otp_monthly["division"] = otp_monthly[otp_div_col].map(otp_div_mapping)
otp_monthly = otp_monthly.groupby(["year_month", "division"]).agg(
    otp_mean=(otp_val_col, "mean"),
).reset_index()

# Aggregate incidents to monthly by division (via line mapping)
incidents_monthly = incidents.copy()
incidents_monthly["year_month"] = incidents_monthly["date"].dt.to_period("M")
incidents_monthly["division"] = incidents_monthly[line_col].apply(
    lambda x: "A" if x in DIVISION_A_LINES else "B" if x in DIVISION_B_LINES else None
)
incidents_monthly = incidents_monthly.dropna(subset=["division"])
incident_counts = incidents_monthly.groupby(["year_month", "division"]).size().reset_index(name="incident_count")

# Aggregate service delivered to monthly by division
service_monthly = service.copy()
service_monthly["year_month"] = service_monthly["date"].dt.to_period("M")
service_monthly["division"] = service_monthly[svc_line_col].apply(
    lambda x: "A" if x in DIVISION_A_LINES else "B" if x in DIVISION_B_LINES else None
)
service_monthly = service_monthly.dropna(subset=["division"])
service_agg = service_monthly.groupby(["year_month", "division"]).agg(
    service_pct_mean=(svc_pct_col, "mean"),
).reset_index()

# Join everything
model_df = mdbf_monthly.merge(otp_monthly, on=["year_month", "division"], how="left")
model_df = model_df.merge(incident_counts, on=["year_month", "division"], how="left")
model_df = model_df.merge(service_agg, on=["year_month", "division"], how="left")

# Fill missing incident counts with 0
model_df["incident_count"] = model_df["incident_count"].fillna(0)

# Add temporal features
model_df["month_num"] = model_df["year_month"].dt.month
model_df["year"] = model_df["year_month"].dt.year
model_df["division_encoded"] = (model_df["division"] == "A").astype(int)

# Lag features for MDBF
for div in ["A", "B"]:
    mask = model_df["division"] == div
    model_df.loc[mask, "mdbf_lag_1m"] = model_df.loc[mask, "mdbf_mean"].shift(1)
    model_df.loc[mask, "mdbf_lag_3m"] = model_df.loc[mask, "mdbf_mean"].shift(3)
    model_df.loc[mask, "mdbf_rolling_3m"] = model_df.loc[mask, "mdbf_mean"].rolling(3, min_periods=1).mean()

# Drop rows with NaN from lags
model_df = model_df.dropna().reset_index(drop=True)

print(f"Joint dataset shape: {model_df.shape}")
print(f"Columns: {list(model_df.columns)}")
model_df.head()

In [ ]:
# ---- Random Forest: Predict Monthly MDBF ----
FEATURE_COLS = [
    "otp_mean", "incident_count", "service_pct_mean",
    "month_num", "year", "division_encoded",
    "mdbf_lag_1m", "mdbf_lag_3m", "mdbf_rolling_3m",
    "mdbf_min", "mdbf_max", "n_car_classes",
]

TARGET = "mdbf_mean"

X = model_df[FEATURE_COLS]
y = model_df[TARGET]

# Temporal split: last 20% as test
split_idx = int(len(model_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

# Evaluate
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = rf_model.score(X_test, y_test)

print(f"\nRandom Forest Results:")
print(f"  RMSE: {rmse:,.0f} miles")
print(f"  MAE:  {mae:,.0f} miles")
print(f"  R2:   {r2:.3f}")

# Feature importance
importance = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Actual vs Predicted
test_dates = model_df["year_month"].iloc[split_idx:].astype(str)
axes[0].plot(test_dates.values, y_test.values, "o-", color="steelblue", label="Actual", markersize=4)
axes[0].plot(test_dates.values, y_pred, "s--", color="red", label="Predicted", markersize=4)
axes[0].set_title(f"MDBF: Actual vs Predicted (R2={r2:.3f})", fontweight="bold")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("MDBF (miles)")
axes[0].tick_params(axis="x", rotation=45)
# Show every 3rd label
for i, label in enumerate(axes[0].xaxis.get_ticklabels()):
    if i % 3 != 0:
        label.set_visible(False)
axes[0].legend()

# Feature importance
importance.plot(kind="barh", ax=axes[1], color="steelblue", edgecolor="navy")
axes[1].set_title("Feature Importance (MDI)", fontweight="bold")
axes[1].set_xlabel("Importance")
axes[1].grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

In [ ]:
# ---- MDBF Time-Series Decomposition for Top Car Classes ----
top_classes = mdbf.groupby(car_class_col)[mdbf_col].count().nlargest(4).index.tolist()

fig, axes = plt.subplots(len(top_classes), 4, figsize=(20, 4 * len(top_classes)), sharex=True)

for row, car_class in enumerate(top_classes):
    subset = mdbf[mdbf[car_class_col] == car_class].set_index("date")[[mdbf_col]].sort_index()
    # Resample to monthly and forward-fill
    monthly = subset.resample("M").mean().ffill()

    if len(monthly) < 24:
        for col in range(4):
            axes[row, col].text(0.5, 0.5, "Insufficient data", ha="center", va="center")
            if col == 0:
                axes[row, col].set_ylabel(car_class, fontsize=11, fontweight="bold")
        continue

    decomp = seasonal_decompose(monthly[mdbf_col], model="additive", period=12)

    components = [
        (decomp.observed, "Observed"),
        (decomp.trend, "Trend"),
        (decomp.seasonal, "Seasonal"),
        (decomp.resid, "Residual"),
    ]

    for col, (data, title) in enumerate(components):
        axes[row, col].plot(data, linewidth=1)
        if row == 0:
            axes[row, col].set_title(title, fontweight="bold")
        if col == 0:
            axes[row, col].set_ylabel(car_class, fontsize=11, fontweight="bold")
        axes[row, col].grid(True, alpha=0.3)

fig.suptitle("MDBF Seasonal Decomposition by Car Class (period=12 months)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Summary & Findings

**Fleet Reliability:**
- Newer car classes (R211, R179) generally show higher MDBF than legacy fleets (R62, R68)
- Division A and Division B show statistically different reliability profiles, driven by fleet composition
- MDBF exhibits seasonal patterns, with dips during summer months (heat-related failures)

**Incident Patterns:**
- Signal failures remain the dominant incident category system-wide
- Certain lines show disproportionate incident rates, suggesting infrastructure-specific issues

**Service Delivery:**
- Most lines deliver 90-98% of scheduled service, with weekday performance slightly higher
- Lines with older infrastructure show lower service delivery percentages

**Predictive Model:**
- The Random Forest model successfully captures MDBF trends using operational features
- Lagged MDBF values and incident counts are the strongest predictors
- The model can serve as an early warning system for reliability degradation

**Next Steps:**
- Incorporate weather data as an external feature for MDBF prediction
- Build line-level (rather than division-level) models for more granular forecasting
- Develop automated alerting when predicted MDBF drops below threshold